# Notebook 05 — Huấn luyện lại B-PIT trên dữ liệu làm giàu và đối chứng M3

**Mục tiêu:** Thực hiện toàn bộ quy trình làm giàu dữ liệu tài chính 9 chỉ số FY2017–2024, thử nghiệm 24 mô hình phân cụm B-PIT (k=3..8), chọn cấu hình tối ưu và chuẩn bị dữ liệu bàn giao cho M3 Backtest.

## Giai đoạn A — Khóa môi trường và xác minh baseline

In [1]:
import subprocess
import sys
from pathlib import Path

# 1. Kiểm tra Git HEAD và trạng thái local
repo_root = Path.cwd()
while not (repo_root / '.git').exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

print('Repository root:', repo_root)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo_root, text=True).strip()
branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=repo_root, text=True).strip()
print(f'Git Branch: {branch}')
print(f'Git HEAD: {head}')

In [2]:
import sqlite3
import numpy as np
import sklearn

# 2. Kiểm tra phiên bản thư viện cốt lõi
print('Python version:', sys.version)
print('SQLite version:', sqlite3.sqlite_version)
print('NumPy version :', np.__version__)
print('Scikit-learn  :', sklearn.__version__)

assert sqlite3.sqlite_version >= '3.35.0', 'SQLite version must support modern syntax'
print('=> Phiên bản thư viện cốt lõi đáp ứng yêu cầu!')

In [3]:
# 3. Chạy pip check kiểm tra tính toàn vẹn phụ thuộc
pip_check = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
print('Pip check return code:', pip_check.returncode)
if pip_check.returncode == 0:
    print('=> Môi trường pip check sạch sẽ, không có xung đột package!')
else:
    print('Pip check warnings/errors:', pip_check.stdout, pip_check.stderr)

In [4]:
# 4. Chạy toàn bộ 52 bài test của M3 baseline để xác minh hệ thống sẵn sàng
import unittest
import os

os.environ['PYTHONPATH'] = str(repo_root / 'src')
loader = unittest.TestLoader()
suite = loader.discover(start_dir=str(repo_root / 'tests' / 'unit'), pattern='test_m3_*.py')
runner = unittest.TextTestRunner(verbosity=1)
result = runner.run(suite)

print(f'Ran {result.testsRun} tests. Errors: {len(result.errors)}, Failures: {len(result.failures)}')
assert result.wasSuccessful(), 'Toàn bộ 52 test M3 phải đạt PASS!'

### Nghiệm thu Giai đoạn A
- **Môi trường:** Đã kiểm tra Python, SQLite, NumPy, scikit-learn.
- **Kiểm thử baseline:** 52/52 bài test của M3 chạy thành công (PASS).
- **Kết luận:** Điều kiện chuyển sang Giai đoạn B (Xác minh và nạp dữ liệu đầu vào) đã ĐẠT!